import sys
import os
import pandas as pd
import py4cytoscape as p4c
import IPython

print("Working directory:", os.getcwd())

In [3]:
import sys
import os
import pandas as pd
import py4cytoscape as p4c
import IPython

print("Working directory:", os.getcwd())

Working directory: /home/jovyan/work/persistent/aop-10


In [5]:
print(f"Loading Javascript client ... {p4c.get_browser_client_channel()} on {p4c.get_jupyter_bridge_url()}")
browser_client_js = p4c.get_browser_client_js()
IPython.display.Javascript(browser_client_js)

Loading Javascript client ... a8dcce59-ff6c-4014-b7d2-9ac407aa939a on https://jupyter-bridge.cytoscape.org


<IPython.core.display.Javascript object>

In [6]:
print(p4c.cytoscape_version_info())
p4c.cytoscape_ping()
print("Connected to Cytoscape")

{'apiVersion': 'v1', 'cytoscapeVersion': '3.10.3', 'automationAPIVersion': '1.13.0', 'py4cytoscapeVersion': '1.13.0', 'jupyterBridgeVersion': '0.0.2'}
You are connected to Cytoscape!
Connected to Cytoscape


In [11]:
from SPARQLWrapper import SPARQLWrapper, JSON
import json

endpoint_url = "https://aopwiki.rdf.bigcat-bioinformatics.org/"

query = """
PREFIX aopo: <http://aopkb.org/aop_ontology#>
PREFIX dc: <http://purl.org/dc/elements/1.1/>

SELECT ?AOP ?AOPName
WHERE {
  ?AOP a aopo:AdverseOutcomePathway ;
       dc:title ?AOPName .
}
"""

sparql = SPARQLWrapper(endpoint_url)
sparql.setQuery(query)
sparql.setReturnFormat(JSON)
sparql.addCustomHttpHeader("Accept", "application/sparql-results+json")

results = sparql.query().convert()

print(type(results))
print(results)

<class 'bytes'>
b'<!DOCTYPE html>\n<html>\n  <head>\n\t<meta charset="utf-8">\n    <meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no">\n    <meta name="description" content="Snorql: A SPARQL Explorer - Extended Edition">\n    <meta name="author" content="Ammar Ammar (https://github.com/ammar257ammar)">\n\t<link rel="icon" href="assets/images/AOP-Wiki RDF logo square large.png">\n\n    <title>AOP-Wiki SNORQL UI</title>\n\t<link rel="stylesheet" type="text/css" href="assets/css/bootstrap.min.css" />\n\t<link rel="stylesheet" type="text/css" href="assets/codemirror/lib/codemirror.css" />\n\t<link rel="stylesheet" type="text/css" href="assets/codemirror/addon/display/fullscreen.css" />\n\t<link rel="stylesheet" type="text/css" href="assets/css/bootstrap-treeview.min.css" />\n    <link rel="stylesheet" type="text/css" href="assets/css/style.css" />\n\n\t<script type="text/javascript" src="assets/js/jquery.min.js"></script>\n    <script type="text/javascrip

In [15]:
import pandas as pd
data = pd.read_csv('aop_10_goprocess.csv')
#data.head() # to display the first 5 lines of loaded data

# Extract the identifier and save it in a new column 'keid'
data['keid'] = data['KE'].apply(lambda x: x.split('/')[-1])
#data['geneid'] = data['id'].apply(lambda x: x.split('/')[-1])

data['go_process'] = data['object'].apply(lambda x: x.split('/')[-1])

# Display the DataFrame
data.head()

data.to_csv("aop-10-list.csv", index=False)  

In [16]:
import os
import re
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from SPARQLWrapper import SPARQLWrapper, JSON
import py4cytoscape as p4c

print("Working directory:", os.getcwd())
pd.set_option("display.max_colwidth", 200)

Working directory: /home/jovyan/work/persistent/aop-10


In [17]:
import IPython

browser_client_js = p4c.get_browser_client_js()
IPython.display.Javascript(browser_client_js)

<IPython.core.display.Javascript object>

In [18]:
print(p4c.cytoscape_version_info())
p4c.cytoscape_ping()

{'apiVersion': 'v1', 'cytoscapeVersion': '3.10.3', 'automationAPIVersion': '1.13.0', 'py4cytoscapeVersion': '1.13.0', 'jupyterBridgeVersion': '0.0.2'}
You are connected to Cytoscape!


'You are connected to Cytoscape!'

In [19]:
endpoint = "https://aopwiki.rdf.bigcat-bioinformatics.org/sparql/"

PREFIXES = """
PREFIX aopo: <http://aopkb.org/aop_ontology#>
PREFIX edam: <http://edamontology.org/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX dc: <http://purl.org/dc/elements/1.1/>
"""

ke_gene_query = PREFIXES + """
SELECT DISTINCT ?keid ?ncbi
WHERE {
    ?ke a aopo:KeyEvent ;
        edam:data_1025 ?object ;
        rdfs:label ?keid .
    ?object skos:exactMatch ?id .
    ?id a edam:data_1027 ;
        edam:data_1027 ?ncbi .
}
"""

aop_go_query = PREFIXES + """
SELECT DISTINCT ?KE ?s ?o
WHERE {
    ?KE a aopo:KeyEvent ;
        <http://purl.obolibrary.org/obo/GO_0008150> ?s .
    ?s a <http://purl.obolibrary.org/obo/GO_0008150> ;
       dc:title ?o .
}
"""

aop_ke_query = PREFIXES + """
SELECT DISTINCT ?keid ?aop_id
WHERE {
    ?ke a aopo:KeyEvent ;
        rdfs:label ?keid .
    ?aop a aopo:AdverseOutcomePathway ;
         rdfs:label ?aop_id ;
         aopo:has_key_event ?ke .
}
"""

In [20]:
def run_sparql(endpoint_url: str, query: str) -> pd.DataFrame:
    sparql = SPARQLWrapper(endpoint_url)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    sparql.addCustomHttpHeader("Accept", "application/sparql-results+json")

    results = sparql.query().convert()

    rows = []
    for row in results["results"]["bindings"]:
        parsed = {k: v["value"] for k, v in row.items()}
        rows.append(parsed)

    return pd.DataFrame(rows)

In [21]:
AOP_KE_query = run_sparql(endpoint, aop_ke_query)
KE_gene_query = run_sparql(endpoint, ke_gene_query)
AOP_GO_query = run_sparql(endpoint, aop_go_query)

print(AOP_KE_query.head())
print(KE_gene_query.head())
print(AOP_GO_query.head())

      keid   aop_id
0  KE 2268  AOP 545
1  KE 2268  AOP 548
2  KE 2269  AOP 545
3  KE 2269  AOP 548
4  KE 2271  AOP 545
      keid    ncbi
0  KE 2268    6721
1  KE 2268    3638
2  KE 2269    6721
3  KE 2269    3638
4  KE 2276  255738
                                        KE  \
0  https://identifiers.org/aop.events/1635   
1  https://identifiers.org/aop.events/2089   
2  https://identifiers.org/aop.events/2244   
3  https://identifiers.org/aop.events/2245   
4  https://identifiers.org/aop.events/2155   

                                                           s  \
0  http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C25830   
1                  http://purl.obolibrary.org/obo/GO_0001649   
2                  http://purl.obolibrary.org/obo/GO_0007166   
3                  http://purl.obolibrary.org/obo/GO_0007166   
4                  http://purl.obolibrary.org/obo/GO_0008146   

                                         o  
0                         DNA Strand Break  
1             

In [22]:
def extract_and_add_prefix(url: str) -> str:
    match = re.search(r"aop\.events/(\d+)", str(url))
    return f"KE{match.group(1)}" if match else None

def extract_value_ont(url: str) -> str:
    url = str(url).rstrip(">")
    if "#" in url:
        return url.split("#")[-1]
    return url.rstrip("/").split("/")[-1]

In [23]:
AOP_GO_query["KE_id"] = AOP_GO_query["KE"].apply(extract_and_add_prefix)
AOP_GO_query["Ontology_id"] = AOP_GO_query["s"].apply(extract_value_ont)

AOP_GO_query_GO = AOP_GO_query[AOP_GO_query["Ontology_id"].str.startswith("GO_", na=False)].copy()
AOP_GO_query_GO["Ontology_id_1"] = AOP_GO_query_GO["Ontology_id"].str.replace("GO_", "GO:", n=1, regex=False)

print(AOP_GO_query_GO.head())

                                        KE  \
1  https://identifiers.org/aop.events/2089   
2  https://identifiers.org/aop.events/2244   
3  https://identifiers.org/aop.events/2245   
4  https://identifiers.org/aop.events/2155   
5  https://identifiers.org/aop.events/2098   

                                           s  \
1  http://purl.obolibrary.org/obo/GO_0001649   
2  http://purl.obolibrary.org/obo/GO_0007166   
3  http://purl.obolibrary.org/obo/GO_0007166   
4  http://purl.obolibrary.org/obo/GO_0008146   
5  http://purl.obolibrary.org/obo/GO_0022008   

                                         o   KE_id Ontology_id Ontology_id_1  
1               osteoblast differentiation  KE2089  GO_0001649    GO:0001649  
2  cell surface receptor signaling pathway  KE2244  GO_0007166    GO:0007166  
3  cell surface receptor signaling pathway  KE2245  GO_0007166    GO:0007166  
4                sulfotransferase activity  KE2155  GO_0008146    GO:0008146  
5                             neurogene

In [24]:
AOP_KE_edges = AOP_KE_query.copy()
AOP_KE_edges.columns = ["source", "target"]   # KE -> AOP in your R object order

KE_gene_edges = KE_gene_query.copy()
KE_gene_edges.columns = ["source", "target"]  # KE -> gene

edges_for_network = pd.concat([AOP_KE_edges, KE_gene_edges], ignore_index=True).drop_duplicates()

In [25]:
node_table = pd.DataFrame({
    "id": pd.concat([
        KE_gene_edges["target"],   # genes
        AOP_KE_edges["source"],    # KEs
        AOP_KE_edges["target"]     # AOs / AOP labels
    ], ignore_index=True)
}).drop_duplicates()

gene_ids = set(KE_gene_edges["target"].astype(str))
ke_ids = set(AOP_KE_edges["source"].astype(str))
ao_ids = set(AOP_KE_edges["target"].astype(str))

def assign_group(node_id: str) -> str:
    node_id = str(node_id)
    if node_id in gene_ids:
        return "Gene"
    if node_id in ke_ids:
        return "KE"
    if node_id in ao_ids:
        return "AO"
    return "Unknown"

node_table["group"] = node_table["id"].astype(str).apply(assign_group)

In [13]:
import pandas as pd
import numpy as np


import py4cytoscape as p4c
p4c.cytoscape_ping()

import os
print('Get current working directory : ', os.getcwd())

from IPython.display import Image
from IPython.display import SVG

pd.set_option('display.max_colwidth', 1)


# Series for group values and their repetitions
values = pd.Series(["Gene", "KE"])
repeats = [8, 3]

# Repeat each value as specified
repeated_series = values.repeat(repeats).reset_index(drop=True)

# Extract unique IDs for genes and KEs
unique_gene_id = pd.unique(data[['id']].values.ravel())
unique_ke_id = pd.unique(data[['keid']].values.ravel())

# Combine the unique IDs
combined_ids = np.concatenate([unique_gene_id, unique_ke_id])

# Repeat the `group` values to match the length of `combined_ids`
repeated_series_adjusted = np.resize(repeated_series, len(combined_ids))

nodes = pd.DataFrame({'id': combined_ids, 'group': repeated_series_adjusted})
    # Create edges dataframe
edges = pd.DataFrame(data={'source': data['id'], 'target': data['keid'], 'interaction': 'interacts', 'weight': 1 })
print(nodes)  
    # Create network
p4c.create_network_from_data_frames(nodes, edges, title= "aop10 network")
    

You are connected to Cytoscape!
Get current working directory :  /home/jovyan/work/persistent/aop-10


KeyError: "None of [Index(['id'], dtype='object')] are in the [columns]"

TypeError: 'function' object is not subscriptable